# Phase 3A: Linear Analysis - Granger Causality

## Objective

Test whether news sentiment **statistically predicts** stock returns using Granger Causality —> a linear, lag-based test that asks:

> *"Does knowing yesterday's sentiment help predict today's return, beyond what return history alone can tell us?"*

## Pipeline Overview

```
Phase 1: Data Preparation & Alignment
    ↓
Phase 2: Pre-Analysis Validation (ADF Stationarity Test)
    ↓
Phase 3: Granger Causality Loop (60 tickers × 2 models × 5 lags)
    ↓
Phase 4: Market Signal Control Test (Broad Market)
    ↓
Phase 5: Reporting & Visualization
    ↓
Output: granger_causality_results.csv
```

## Mathematical Foundation

**Granger Causality** tests whether time series X "Granger-causes" Y:

$$Y_t = \alpha + \sum_{i=1}^{p} \beta_i Y_{t-i} + \sum_{i=1}^{p} \gamma_i X_{t-i} + \epsilon_t$$

**Null Hypothesis ($H_0$)**: $\gamma_1 = \gamma_2 = ... = \gamma_p = 0$ (sentiment adds no predictive power)

**Reject $H_0$** if $p < 0.05$ → Sentiment Granger-causes returns

---

## 1. Setup: Imports and Paths

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.stattools import adfuller, grangercausalitytests
import statsmodels.api as sm

import yfinance as yf

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import os
import random
from tqdm import tqdm

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
project_root = '/content/drive/MyDrive/market-sentiment-impact-analysis'

data_processed = os.path.join(project_root, 'data', 'processed')

print(f'Project Root: {project_root}')
print(f'Processed Data: {data_processed}')

---

## 2: Data Preparation & Alignment

### 2.1 Load Data